# Parse Human Comments

Splits each UICrit comment into `expected_standard`, `observed_issue`, `suggested_fix` and `bounding_box`,
applies the manual corrections from `data/processed/2_parsed/manual_comment_parsing_corrections.json`, and saves
`data/processed/2_parsed/uicrit_notna_parsed.parquet` (input of notebook 3).

In [1]:
import re
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils" / "paths.py").exists())
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
from utils.paths import PARSED_DIR, PREPROCESSED_DIR

uicrit_df = pd.read_parquet(PREPROCESSED_DIR / "uicrit_notna.parquet")
print("Shape:", uicrit_df.shape)
uicrit_df.head(2)

Shape: (2955, 11)


,screen_id,app_category,app,screen_task_id,task,aesthetics_rating,learnability,efficiency,usability_rating,design_quality_rating,comments
0,15,Health & Fitness,Seven - 7 Minute Workout,15_T01,Plan and Start Full Body Workouts,6,3,3,6,6,[Comment 1\nThe expected standard is that the ...
1,15,Health & Fitness,Seven - 7 Minute Workout,15_T02,Tap on the Plus icon or explore the Workout Pl...,6,3,3,6,6,[Comment 1\nThe expected standard is to make t...


# Examine Missing Parts of Comments
Count comments lacking each part's primary marker, then drop tasks covered by known variant phrasings.

In [2]:
def find_comments(pattern, comments=None, missing=False):
    """(screen_task_id, comment index, comment) for comments that match `pattern` (or don't, if missing=True)."""
    if comments is None:
        comments = [(row.screen_task_id, i, c) for row in uicrit_df.itertuples() for i, c in enumerate(row.comments)]
    return [item for item in comments if bool(re.search(pattern, item[2].lower())) != missing]

def drop_variants(missing, variants):
    """Remove tasks that use a variant phrasing from the missing list, printing each variant's count."""
    for label, pattern in variants.items():
        found = find_comments(pattern, missing if label == "the current design" else None)
        print(f"Total '{label}' comments:", len(found))
        task_ids = {item[0] for item in found}
        missing = [item for item in missing if item[0] not in task_ids]
    print("Total missing after removing variants:", len(missing))
    return missing

## Examine Expected Standard

In [3]:
missing_standard = find_comments(r"the\s+expected\s+standard\s+is", missing=True)
print("Total missing expected standard comments:", len(missing_standard))
missing_standard = drop_variants(missing_standard, {
    "the standard is": r"the standard is",
    "\\nhe expected standard is": r"\nhe expected standard is",
})

Total missing expected standard comments: 85
Total 'the standard is' comments: 16
Total '\nhe expected standard is' comments: 39
Total missing after removing variants: 30


**Expected standard phrasings**
- Covered by parser variants:
  - _The standard is_: 2168_T03, 9560_T01, 14596_T03, 23151_T02, 23858_T02, 28483_T01, 29243_T03, 39697_T01, 47284_T03, 48615_T01, 51772_T01, 53276_T03, 56085_T02, 63936_T01, 64788_T02, 65444_T02
- Not matched by the parser:
  - _Expected standards_: 23441_T01 (3 comments)
  - _The expected tandard is_ (typo): 10112_T02, 20767_T03
  - Phrase missing: 72017_T03

**Malformed comments** (also missing the other parts)
- Only has a fix: 18604_T01, 33936_T02
- 4439_T01: "a Game of Memory Grid"
- 32469_T01: "Consider making their visual appearance similar."
- 64258_T03: "adding missing information"

## Examine Current Design

In [4]:
missing_current_design = find_comments(r"in\s+the\s+current\s+design", missing=True)
print("Total missing current design comments:", len(missing_current_design))
missing_current_design = drop_variants(missing_current_design, {
    "in the current layout": r"in the current layout",
    "in this current design": r"in this current design",
    "in the current ,": r"in\s+the\s+current ,",
    "in current design": r"in\s+current\s+design",
    "the current design": r"the\s+current\s+design",  # searched within the remaining missing comments only
})

Total missing current design comments: 82
Total 'in the current layout' comments: 33
Total 'in this current design' comments: 22
Total 'in the current ,' comments: 4


Total 'in current design' comments: 4
Total 'the current design' comments: 9
Total missing after removing variants: 10


**Current design phrasings**
- Covered by parser variants:
  - _In the current ,_: 15707_T02, 30487_T02, 38193_T03, 57973_T03
- Not matched by the parser:
  - _In the current shopping app design,_: 45949_T02
  - _In the current text,_: 59840_T03
  - _current design,_: 34517_T01
  - Current design described without the phrase: 9960_T02, 43000_T02
- Plus the malformed comments listed under Expected Standard.

## Examine Suggested Fix

In [5]:
missing_fix = find_comments(r"to\s+fix\s+this,", missing=True)
print("Total missing to fix this comments:", len(missing_fix))
missing_fix = drop_variants(missing_fix, {
    "to fix it": r"to\s+fix\s+it",
    "to fix this issue": r"to\s+fix\s+this\s+issue",
})

Total missing to fix this comments: 95


Total 'to fix it' comments: 15
Total 'to fix this issue' comments: 29
Total missing after removing variants: 51


**Suggested fix phrasings**
- Not matched by the parser (fix markers are strict: only _to fix this_ / _to fix it_):
  - _To Fix,_: 25926_T01, 36768_T02, 68681_T02
  - _To fix ,_: 69946_T03
  - _To address this,_: 32379_T02, 44026_T03, 45328_T01, 48697_T02
  - _To fix the current design,_: 62249_T01
  - Fix suggested without a marker phrase: 66266_T02, 69109_T01
  - Fix missing: 10089_T01, 15321_T03, 46658_T01, 60201_T02
- Plus the malformed comments listed under Expected Standard (18604_T01 and 33936_T02 contain only a fix, without a marker phrase).

## Bounding Box

In [6]:
missing_boundary = find_comments(r"\nbounding box: ", missing=True)
print("Total missing Bounding Box comments:", len(missing_boundary))

Total missing Bounding Box comments: 0


# Parse Comments to Structured Dicts

## Automatically parse all comments

In [7]:
# Comment header, e.g. "Comment 3" or "LLM Comment 2"
RE_COMMENT_ID = re.compile(r"^\s*((?:llm\s+)?comment\s*[:#]?\s*\d+)\b[\s:,\-–—]*", re.I)
RE_BBOX = re.compile(r"(?:\bbounding\s*box\b|\bboundingbox\b)\s*[:\-]?\s*\[([^\]]+)\]", re.I)

# Expected standard markers (primary + variants found above)
EXP_MARKERS = [
    re.compile(r"\bthe\s+expected\s+standard\b", re.I),
    re.compile(r"\bthe\s+standard\s+is\b", re.I),
    re.compile(r"\bhe\s+expected\s+standard\s+is\b", re.I),  # typo: missing 't'
    re.compile(r"\nhe\s+expected\s+standard\s+is\b", re.I),
]
# Observed issue markers (primary + variants found above)
OBS_MARKERS = [
    re.compile(r"\bin\s+the\s+current\s+design\b", re.I),
    re.compile(r"\bin\s+the\s+current\s+layout\b", re.I),
    re.compile(r"\bin\s+this\s+current\s+design\b", re.I),
    re.compile(r"\bin\s+the\s+current\s+,", re.I),
]
# Fix markers (strict: only "to fix this" / "to fix it")
FIX_MARKERS = [
    re.compile(r"\bto\s+fix\s+this\b", re.I),
    re.compile(r"\bto\s+fix\s+it\b", re.I),
]
PART_FIELDS = ["expected_standard", "observed_issue", "suggested_fix"]


def _cleanup_segment(s):
    s = re.sub(r"^[\s,:\-–—]+", "", (s or "").strip())
    return re.sub(r"[\s,:\-–—]+$", "", s.strip()).strip()

def _find_earliest(text, patterns):
    matches = [m for m in (p.search(text) for p in patterns) if m]
    return min(matches, key=lambda m: m.start()) if matches else None

def _remove_expected_glue(seg):
    seg = seg.lstrip(" \t\r\n:,-–—")
    for glue in [r"is\s+that", r"is\s+to", r"that", r"to", r"is"]:  # in this priority
        seg = re.sub(rf"^({glue})\b\s*", "", seg, flags=re.I)
    return _cleanup_segment(seg)

def _parse_label(text, position):
    """Return (comment_label, comment_source, rest of text)."""
    m = RE_COMMENT_ID.match(text)
    if not m:
        return f"Comment {position}", "Human", text.strip()
    cid = m.group(1).strip()
    source = "LLM" if re.match(r"^llm\b", cid, flags=re.I) else "Human"
    number = re.search(r"comment\s*(\d+)", cid, flags=re.I)
    if number:
        label = f"Comment {number.group(1)}"
    else:
        label = re.sub(r"^llm\s+", "", cid[0:1].upper() + cid[1:], flags=re.I).strip()
    return label, source, text[m.end():].strip()

def _extract_bbox(text):
    m = RE_BBOX.search(text)
    if not m:
        return text, None
    nums = [float(x) for x in re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", m.group(1))]
    text = (text[:m.start()] + " " + text[m.end():]).strip()
    if len(nums) < 4:
        return text, nums or None
    return text, dict(zip(["x_min", "y_min", "x_max", "y_max"], nums))

def _extract_fix(text):
    """Text after the earliest fix marker, minus a leading "issue(s)" and comma."""
    m = _find_earliest(text, FIX_MARKERS)
    if not m:
        return None
    rest = re.sub(r"^(issue|issues)\b", "", text[m.end():].lstrip(), flags=re.I).lstrip()
    if rest.startswith(","):
        rest = rest[1:]
    return _cleanup_segment(rest) or None


def parse_single_comment(comment, position):
    raw = str(comment).strip()
    label, source, text = _parse_label(raw, position)
    text, bbox = _extract_bbox(text)  # bbox first so it doesn't pollute the parts

    markers = [(kind, m) for kind, m in [
        ("expected", _find_earliest(text, EXP_MARKERS)),
        ("observed", _find_earliest(text, OBS_MARKERS)),
        ("fix", _find_earliest(text, FIX_MARKERS)),
    ] if m]
    markers.sort(key=lambda x: x[1].start())

    parts = {"expected_standard": None, "observed_issue": None, "suggested_fix": _extract_fix(text)}
    for i, (kind, m) in enumerate(markers):
        seg = text[m.end(): markers[i + 1][1].start() if i + 1 < len(markers) else len(text)]
        if kind == "expected":
            parts["expected_standard"] = _remove_expected_glue(seg) or None
        elif kind == "observed":
            parts["observed_issue"] = _cleanup_segment(seg) or None
    if not markers:  # nothing recognized: keep everything as the observed issue
        parts["observed_issue"] = _cleanup_segment(text) or None

    # Trim any "to fix this/it" tail left inside the observed issue
    if parts["observed_issue"]:
        m = _find_earliest(parts["observed_issue"], FIX_MARKERS)
        if m:
            parts["observed_issue"] = _cleanup_segment(parts["observed_issue"][:m.start()]) or None

    return {
        "comment_label": label,
        "comment_source": source,
        "bounding_box": bbox,
        **parts,
        "raw_text": re.sub(r"\s+", " ", raw.replace(" ", " ")).strip(),
    }


uicrit_df["parsed_comments"] = uicrit_df["comments"].apply(
    lambda comments: [parse_single_comment(c, i) for i, c in enumerate(comments, start=1)]
)

## Apply manual corrections
Parts the parser got wrong were fixed by hand during review; those fixes live in `manual_comment_parsing_corrections.json`
(`null` = part not present). `missing_parts` is computed after corrections.

In [8]:
corrections = json.load(open(PARSED_DIR / "manual_comment_parsing_corrections.json"))
parsed_by_task = dict(zip(uicrit_df["screen_task_id"], uicrit_df["parsed_comments"]))
for c in corrections:
    parsed_by_task[c["screen_task_id"]][c["comment_index"]][c["field"]] = c["value"]
print("Corrections applied:", len(corrections))

for comments in uicrit_df["parsed_comments"]:
    for d in comments:
        d["missing_parts"] = [f for f in PART_FIELDS if not (d[f] and d[f].strip())]

Corrections applied: 158


# Save Parsed Comments

In [10]:
uicrit_df.to_parquet(PARSED_DIR / "uicrit_notna_parsed.parquet", index=False)
uicrit_df["parsed_comments"].iloc[0][0]

{'comment_label': 'Comment 1',
 'comment_source': 'Human',
 'bounding_box': {'x_min': 0.1559446,
  'y_min': 0.05137866,
  'x_max': 0.40545597,
  'y_max': 0.09649163},
 'expected_standard': 'the text’s visual treatment and formatting should make it easy to understand.',
 'observed_issue': 'the text (Workouts) is small even when it is heading.',
 'suggested_fix': 'increase font size and weight to make it  look like a heading',
 'raw_text': 'Comment 1 The expected standard is that the text’s visual treatment and formatting should make it easy to understand. In the current design, the text (Workouts) is small even when it is heading. To fix this, increase font size and weight to make it look like a heading Bounding Box: [0.1559446, 0.05137866, 0.40545597, 0.09649163]',
 'missing_parts': []}